# just_tx vs just_tx_hal：RTT（~5% 丢包工作点）

同一块 ESP32-C3、同一 AP、同一台 TR3000。比较两条**回发**路径在「丢包率约 5%」时的最大可持续发包速率下的往返时延。

| 臂 | 固件 / 冷表 | 回包形状（TC 配对） | 设备计数 |
|---|---|---|---|
| **HAL** | `just_tx_hal` | UDP:8080 原载荷回发 → TC `bounce_ingress`（`sport=8080`） | 串口 `rxcnt 17` |
| **XDP** | `just_drop` + 冷表 `just_tx.elf` | MAC 交换 `XDP_TX` → TC `bounce_ingress`（L2-only） | map `rxcnt[17]` |

**刺激与配对**：每个 UDP 载荷是 `FWSTIM` + `u64` LE seq。TR3000 按固定 `target_pps` 管道式发送（不等待上一包），窗口结束后用 TC 的 T0/`t_bounce` 按 seq 配对。两臂回包在 TC 上均记 bounce（不是 `dport=reply_port` 的 STA echo）。

**工作点标定（每臂一次）**：对数网格扫主机发包速率，再在最优点附近微调。目标丢包率 `loss = 1 − ok/sent ≈ 5%`（接受带约 3.5%–7%）。在带内取**最大** `target_pps`；否则取最接近 5% 的点。

**三个指标**（柱顶标题即表头）：

1. **Median RTT** — 配对成功样本的 `(T1−T0)` 中位数（ms）。无回包的 seq 不进 RTT。
2. **Loss rate** — `1 − ok/sent`。虚线标 5%。标定后各轮应落在附近。
3. **Return PPS** — `ok / window`（主机侧配对成功回包速率）。

另记录 `target_pps`（标定后的主机发包速率）、`device_rxcnt_delta`（串口设备侧计数增量，与 `ok` 对账）。

数据：`formal.just_tx_rtt_*`，`source_note = xpost-test just-tx-rtt`。默认每臂标定后 3 轮、每轮 20 s。

复现：Mac `cargo xtask agent --listen 0.0.0.0:9240 --no-monitor`；TR3000 `./xpost-test`（`[default_bench]=just-tx-rtt`，勿加 `--no-pg`）。本笔记只读库。满速处理 PPS 见 `just_tx_pps.ipynb`。

同一块 ESP32-C3、同一 AP、同一台 TR3000。比较两条**回发**路径在「丢包率约 5%」时的最大可持续发包速率下的往返时延。

| 臂 | 固件 / 冷表 | 回包形状（TC 配对） | 设备计数 |
|---|---|---|---|
| **HAL** | `just_tx_hal` | STA UDP:8080 原载荷回发 → `t_echo` | 串口 `rxcnt 17` |
| **XDP** | `just_drop` + 冷表 `just_tx.elf` | 只交换 MAC 后 `XDP_TX` → `t_bounce` | map `rxcnt[17]` |

每个 UDP 载荷是 `FWSTIM` + `u64` LE seq。TR3000 按固定 `target_pps` 管道式发送（不等待上一包），窗口结束后用 TC 的 T0/T1 按 seq 配对。HAL 回包与 XDP MAC-swap 在 TC 上都记为 `bounce_ingress`（`sport==8080` 且 `dport!=reply_port`，或 L2-only bounce）。

**工作点标定（每臂一次）**：在区间内二分搜索主机发包速率，目标丢包率 `loss = 1 − ok/sent ≈ 5%`（接受带 4%–6%）。取最接近 5% 的速率；若并列偏好更高速率且 `loss ≤ 6%`。标定窗短于测量窗。

**三个指标**（柱顶标题即表头）：

1. **Median RTT** — 配对成功样本的 `(T1−T0)` 中位数（ms）。无回包的 seq 不进 RTT。
2. **Loss rate** — `1 − ok/sent`。虚线标 5%。标定后各轮应落在附近。
3. **Return PPS** — `ok / window`（主机侧配对成功回包速率）。

另记录 `target_pps`（标定后的主机发包速率）、`device_rxcnt_delta`（串口读出的设备侧计数增量，与 `ok` 对账）。

数据：`formal.just_tx_rtt_*`，`source_note = xpost-test just-tx-rtt`。默认每臂标定后 3 轮、每轮 20 s。

复现：Mac `cargo xtask agent --listen 0.0.0.0:9240 --no-monitor`；TR3000 `./xpost-test`（`[default_bench]=just-tx-rtt`，勿加 `--no-pg`）。本笔记只读库。满速处理 PPS 见 `just_tx_pps.ipynb`。


In [ ]:
from collections import defaultdict

import matplotlib.pyplot as plt
from IPython.display import display

from utils.db import dsn_summary
from utils.experiment import (
    figure1_tx_rtt,
    load_just_tx_rtt_runs,
    mean_t_ci,
)

SOURCE_NOTE = "xpost-test just-tx-rtt"

plt.ioff()
plt.close("all")
print(dsn_summary())

runs = load_just_tx_rtt_runs(run_kind="pilot", source_note=SOURCE_NOTE)
assert len(runs), (
    "formal.just_tx_rtt_runs 无 pilot 行：先在 TR3000 跑 ./xpost-test"
    "（[default_bench]=just-tx-rtt，勿加 --no-pg），且 .env 设 PGSCHEMA=formal"
)
print(f"runs={len(runs)}")

by = defaultdict(list)
for _, row in runs.iterrows():
    by[row["variant"]].append(row)


def _col(rows, key):
    return [float(r[key]) for r in rows if r[key] is not None and r[key] == r[key]]


summaries = {}
for v, rows in by.items():
    rtt = _col(rows, "median_rtt_ms")
    loss = _col(rows, "loss_rate")
    rpps = _col(rows, "return_pps")
    tpps = _col(rows, "target_pps")
    cal = _col(rows, "calibrated_loss")
    _, rtt_lo, rtt_hi = mean_t_ci(rtt)
    _, loss_lo, loss_hi = mean_t_ci(loss)
    _, pps_lo, pps_hi = mean_t_ci(rpps)
    summaries[v] = {
        "median_rtt": sum(rtt) / len(rtt) if rtt else float("nan"),
        "rtt_ci": (rtt_lo, rtt_hi),
        "loss": sum(loss) / len(loss) if loss else float("nan"),
        "loss_ci": (loss_lo, loss_hi),
        "return_pps": sum(rpps) / len(rpps) if rpps else float("nan"),
        "pps_ci": (pps_lo, pps_hi),
        "target_pps": sum(tpps) / len(tpps) if tpps else float("nan"),
        "cal_loss": sum(cal) / len(cal) if cal else float("nan"),
        "n": len(rows),
        "per_rtt": [round(x, 2) for x in rtt],
        "per_loss": [round(100 * x, 2) for x in loss],
    }
    s = summaries[v]
    print(
        f"{v}: n={s['n']}  target_pps≈{s['target_pps']:.0f}  cal_loss≈{100*s['cal_loss']:.1f}%  "
        f"median_RTT={s['median_rtt']:.2f}ms  loss={100*s['loss']:.2f}%  "
        f"return_pps={s['return_pps']:.0f}  per_rtt={s['per_rtt']}  per_loss%={s['per_loss']}"
    )

hal, xdp = summaries.get("hal", {}), summaries.get("xdp", {})
fig, _ = figure1_tx_rtt(
    hal_median_rtt_ms=hal.get("median_rtt"),
    xdp_median_rtt_ms=xdp.get("median_rtt"),
    hal_rtt_ci=hal.get("rtt_ci"),
    xdp_rtt_ci=xdp.get("rtt_ci"),
    hal_loss=hal.get("loss"),
    xdp_loss=xdp.get("loss"),
    hal_loss_ci=hal.get("loss_ci"),
    xdp_loss_ci=xdp.get("loss_ci"),
    hal_return_pps=hal.get("return_pps"),
    xdp_return_pps=xdp.get("return_pps"),
    hal_pps_ci=hal.get("pps_ci"),
    xdp_pps_ci=xdp.get("pps_ci"),
    title="just_tx HAL vs XDP — RTT @ ~5% loss",
)
display(fig)
plt.close(fig)
